# SHViT + Caltech-101 on Google Colab

This notebook walks through:
1. Enabling GPU and checking the environment
2. Cloning SHViT and installing dependencies
3. Downloading pretrained SHViT-S4 weights
4. Downloading Caltech-101 with Tip-Adapter style preprocessing
5. Verifying the model loads and runs inference
6. Running SHViT's official eval script

> **Before running:** Go to `Runtime → Change runtime type → T4 GPU`
> All outputs from this stage are saved under `/content/CV_Research_Paper_Caltech101/`.


## 0. Check GPU & environment

In [1]:
import torch

print('PyTorch version :', torch.__version__)
print('CUDA available  :', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU             :', torch.cuda.get_device_name(0))
    print('VRAM            :', round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1), 'GB')

import sys
print('Python version  :', sys.version.split()[0])

PyTorch version : 2.10.0+cu128
CUDA available  : True
GPU             : NVIDIA RTX PRO 6000 Blackwell Server Edition
VRAM            : 102.0 GB
Python version  : 3.12.13


## 1. (Optional) Mount Google Drive

Caltech-101 is only ~150 MB so this is mostly for persisting outputs across
sessions. Mounting Drive lets you keep the dataset and downloaded weights
between Colab restarts. Skip this cell if you are happy to re-download.


In [2]:
USE_DRIVE = False   # set True to persist data + outputs in Google Drive

OUT_ROOT = '/content/CV_Research_Paper_Caltech101'

if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    DATA_ROOT = '/content/drive/MyDrive/caltech101_data'
    OUT_ROOT  = '/content/drive/MyDrive/CV_Research_Paper_Caltech101'
else:
    DATA_ROOT = '/content/caltech101_data'

import os
os.makedirs(OUT_ROOT, exist_ok=True)
os.makedirs(DATA_ROOT, exist_ok=True)

print('Dataset will be stored at:', DATA_ROOT)
print('Outputs will be written under:', OUT_ROOT)


Dataset will be stored at: /content/caltech101_data
Outputs will be written under: /content/CV_Research_Paper_Caltech101


## 2. Clone SHViT

In [3]:
import os

if not os.path.isdir('/content/SHViT'):
    !git clone https://github.com/ysj9909/SHViT.git /content/SHViT
else:
    print('SHViT already cloned, skipping.')

!ls /content/SHViT

Cloning into '/content/SHViT'...
remote: Enumerating objects: 183, done.
remote: Counting objects: 100% (183/183), done.
remote: Compressing objects: 100% (152/152), done.
remote: Total 183 (delta 84), reused 82 (delta 27), pack-reused 0 (from 0)
Receiving objects: 100% (183/183), 168.52 KiB | 8.43 MiB/s, done.
Resolving deltas: 100% (84/84), done.
acc_vs_thro.png  engine.py	  losses.py  README.md	       utils.py
data		 export_model.py  main.py    requirements.txt
downstream	 LICENSE	  model      speed_test.py


## 3. Install dependencies

Colab ships with PyTorch 2.x which satisfies SHViT's `>=1.11` requirement,
so we only need to install the extra packages from `requirements.txt`.

`--no-deps` on timm avoids overwriting Colab's torch/torchvision with
the older versions timm 0.5.4 would otherwise pull in.

In [4]:
# scikit-image==0.19.3 from SHViT's requirements has no wheels for Python
# 3.12 (Colab's default) — and we don't actually need it. Install only what
# the SHViT model architecture needs.
!pip install -q timm==0.5.4 --no-deps
!pip install -q einops==0.4.1 easydict
print('Dependencies installed.')

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 431.5/431.5 kB 20.0 MB/s eta 0:00:00
Dependencies installed.


## 4. Download SHViT-S4 pretrained weights

In [5]:
WEIGHTS_DIR = '/content/weights'
WEIGHTS_PATH = f'{WEIGHTS_DIR}/shvit_s4.pth'

os.makedirs(WEIGHTS_DIR, exist_ok=True)

if not os.path.exists(WEIGHTS_PATH):
    !wget -q --show-progress \
        https://github.com/ysj9909/SHViT/releases/download/v1.0/shvit_s4.pth \
        -O {WEIGHTS_PATH}
else:
    print('Weights already downloaded, skipping.')

size_mb = os.path.getsize(WEIGHTS_PATH) / 1e6
print(f'Checkpoint size: {size_mb:.1f} MB  ->  {WEIGHTS_PATH}')

/content/weights/sh 100%[===================>] 254.37M   175MB/s    in 1.5s    
Checkpoint size: 266.7 MB  ->  /content/weights/shvit_s4.pth


## 5. Download Caltech-101 with Tip-Adapter style preprocessing

This clones our Caltech-101 project and runs `prepare_caltech101.py`, which
downloads the dataset via `torchvision.datasets.Caltech101` and prepares a
Tip-Adapter style `split_zhou_Caltech101.json` (using gdown if available,
otherwise generating a deterministic 50/20/30 train/val/test split).


In [6]:
import os, shutil

REPO_DIR = '/content/Vision_Project_spring_26'
if not os.path.isdir(REPO_DIR):
    !git clone -b Vision_Project_spring_26_Caltech101 \
        https://github.com/saif-farid-tech/Vision_Project_spring_26.git {REPO_DIR}

# Make sure the prepare script + dataset helpers are importable from /content
for fname in [
    'prepare_caltech101.py',
    'splits.py',
    'metrics.py',
    'augmentation.py',
]:
    shutil.copy(f'{REPO_DIR}/{fname}', f'/content/{fname}')

# Vendor the datasets/ package (Tip-Adapter utilities)
DATASETS_DST = '/content/datasets'
if os.path.isdir(DATASETS_DST):
    shutil.rmtree(DATASETS_DST)
shutil.copytree(f'{REPO_DIR}/datasets', DATASETS_DST)

!pip install -q gdown
!python /content/prepare_caltech101.py --root {DATA_ROOT}


Cloning into '/content/Vision_Project_spring_26'...
remote: Enumerating objects: 485, done.
remote: Counting objects: 100% (170/170), done.
remote: Compressing objects: 100% (98/98), done.
remote: Total 485 (delta 82), reused 131 (delta 63), pack-reused 315 (from 2)
Receiving objects: 100% (485/485), 412.60 MiB | 44.06 MiB/s, done.
Resolving deltas: 100% (200/200), done.
[caltech101] downloading Caltech-101 via torchvision into /content/caltech101_data ...
100% 137M/137M [00:11<00:00, 11.9MB/s]
[caltech101] moving /content/caltech101_data/caltech101/101_ObjectCategories -> /content/caltech101_data/caltech-101/101_ObjectCategories
[caltech101] trying official split download via gdown: https://drive.google.com/uc?id=1hyarUivQE36mY6jSomru6Fjd-JzwcCzN
Downloading...
From: https://drive.google.com/uc?id=1hyarUivQE36mY6jSomru6Fjd-JzwcCzN
To: /content/caltech101_data/caltech-101/split_zhou_Caltech101.json
100% 809k/809k [00:00<00:00, 153MB/s]
[caltech101] official split saved to /content/calt

In [7]:
# Sanity check
from pathlib import Path
img_root = Path(DATA_ROOT) / 'caltech-101' / '101_ObjectCategories'
split_json = Path(DATA_ROOT) / 'caltech-101' / 'split_zhou_Caltech101.json'

n_class = sum(1 for p in img_root.iterdir() if p.is_dir())
n_img   = sum(1 for _ in img_root.rglob('*.jpg'))
print(f'On-disk:  {n_class} category folders, {n_img} images at {img_root}')
print(f'Split    : {split_json} (exists: {split_json.exists()})')


On-disk:  102 category folders, 9144 images at /content/caltech101_data/caltech-101/101_ObjectCategories
Split    : /content/caltech101_data/caltech-101/split_zhou_Caltech101.json (exists: True)


## 6. Verify model loads and runs inference

Loads the SHViT-S4 checkpoint and runs 50 Caltech-101 images through it.
Predictions are ImageNet class indices (not Caltech-101 labels) — accuracy
will be ~zero until the model is fine-tuned. The goal here is just to
confirm no import / shape errors occur.


In [8]:
import sys, time, pathlib
import torch
from PIL import Image
from torchvision import transforms
from torchvision.transforms import InterpolationMode

sys.path.insert(0, '/content/SHViT')

from model import shvit
import timm

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Using device:', DEVICE)

model_shvit = timm.create_model('shvit_s4', pretrained=False, num_classes=1000)

ckpt = torch.load(WEIGHTS_PATH, map_location='cpu', weights_only=False)
state_dict = ckpt.get('model', ckpt)
missing, unexpected = model_shvit.load_state_dict(state_dict, strict=False)
print(f'Missing keys: {len(missing)}   Unexpected keys: {len(unexpected)}')

model_shvit.to(DEVICE).eval()
print('Model loaded successfully.')


Using device: cuda
Missing keys: 0   Unexpected keys: 0
Model loaded successfully.


In [9]:
NUM_IMAGES = 50

# CLIP / Tip-Adapter normalization
CLIP_MEAN = (0.48145466, 0.4578275, 0.40821073)
CLIP_STD  = (0.26862954, 0.26130258, 0.27577711)

transform = transforms.Compose([
    transforms.Resize(256, interpolation=InterpolationMode.BICUBIC),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(CLIP_MEAN, CLIP_STD),
])

image_dir = pathlib.Path(DATA_ROOT) / 'caltech-101' / '101_ObjectCategories'
SKIP = {'BACKGROUND_Google', 'Faces_easy'}
items = []
for cls_dir in sorted(image_dir.iterdir()):
    if not cls_dir.is_dir() or cls_dir.name in SKIP:
        continue
    for img_path in sorted(cls_dir.glob('*.jpg')):
        items.append((img_path, cls_dir.name))
        if len(items) >= NUM_IMAGES:
            break
    if len(items) >= NUM_IMAGES:
        break

print(f'Running inference on {len(items)} images ...')
t0 = time.perf_counter()
results = []
with torch.no_grad():
    for img_path, true_class in items:
        x = transform(Image.open(img_path).convert('RGB')).unsqueeze(0).to(DEVICE)
        pred = int(model_shvit(x).argmax(1).item())
        results.append((img_path.name, true_class, pred))

elapsed = time.perf_counter() - t0
print(f'\n{"Image":<30} {"True class":<25} {"Pred idx":>8}')
print('-' * 65)
for name, cls, pred in results[:15]:
    print(f'{name:<30} {cls:<25} {pred:>8}')
print(f'\nTotal: {elapsed:.2f}s  ({elapsed/len(results)*1000:.1f} ms/image)')
print('\n[OK] Model ran without errors.')


Running inference on 50 images ...

Image                          True class                Pred idx
-----------------------------------------------------------------
image_0001.jpg                 Faces                          453
image_0002.jpg                 Faces                          453
image_0003.jpg                 Faces                            0
image_0004.jpg                 Faces                          433
image_0005.jpg                 Faces                          610
image_0006.jpg                 Faces                          678
image_0007.jpg                 Faces                          419
image_0008.jpg                 Faces                          838
image_0009.jpg                 Faces                          419
image_0010.jpg                 Faces                          838
image_0011.jpg                 Faces                          610
image_0012.jpg                 Faces                          855
image_0013.jpg                 Faces    

## 7. Run SHViT's official eval script

The Caltech-101 image directory follows the ImageNet-style class folders
layout that SHViT's `--data-set IMNET` accepts. Expect ~zero accuracy
relative to ImageNet-1K classes — fine-tuning happens in Stage 3.


In [10]:
import re

with open('/content/SHViT/main.py', 'r') as f:
    content = f.read()
content = re.sub(r"torch\.load\(([^,]+),\s*map_location='cpu'\)",
                 r"torch.load(\1, map_location='cpu', weights_only=False)",
                 content)
with open('/content/SHViT/main.py', 'w') as f:
    f.write(content)

# SHViT main.py expects --data-path to contain train/ and val/ subdirs.
# Caltech-101 has a single 101_ObjectCategories/ folder, so for this sanity
# run we point both train and val at the same directory.
import os, shutil
IM_ROOT = '/content/imnet_caltech_sanity'
os.makedirs(IM_ROOT, exist_ok=True)
for split in ('train', 'val'):
    link = f'{IM_ROOT}/{split}'
    if not os.path.exists(link):
        os.symlink(f'{DATA_ROOT}/caltech-101/101_ObjectCategories', link)

!python /content/SHViT/main.py \
    --model shvit_s4 \
    --eval \
    --resume {WEIGHTS_PATH} \
    --data-path {IM_ROOT} \
    --data-set IMNET \
    --batch-size 64 \
    --num_workers 2 \
    --device cuda


Not using distributed mode
Creating model: shvit_s4
number of params: 16588484
/usr/local/lib/python3.12/dist-packages/timm/utils/cuda.py:40: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  self._scaler = torch.cuda.amp.GradScaler()
Loading local checkpoint at /content/weights/shvit_s4.pth
<All keys matched successfully>
Evaluating model: shvit_s4
/content/SHViT/engine.py:91: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():
Test:  [ 0/96]  eta: 0:03:17  loss: 8.5118 (8.5118)  acc1: 0.0000 (0.0000)  acc5: 0.0000 (0.0000)  time: 2.0623  data: 0.2691  max mem: 1013
Test:  [10/96]  eta: 0:00:21  loss: 8.4703 (8.4073)  acc1: 0.0000 (0.0000)  acc5: 0.0000 (0.0000)  time: 0.2448  data: 0.0747  max mem: 1013
Test:  [20/96]  eta: 0:00:12  loss: 8.4290 (8.4477)  acc1: 0.0000 (0.0000)  acc5: 0.0000 (0.0000)  ti

## Next steps — fine-tuning on Caltech-101

To actually train SHViT on Caltech-101, head over to Stage 3 and run
`finetune_shvit_caltech101.py`, which uses the Tip-Adapter split JSON,
CLIP normalization, RandAugment / RandomErasing / Mixup / CutMix /
label-smoothing, AGC-style gradient clipping, and cosine LR with warmup
(the same recipe as the SHViT paper, but with `--nb_classes 100`).
